# 2503-CSCI444 HW3: Transformer Language Models

## General Instructions

This assignment deals with pretrained language models, inference and tokenization. PLEASE START EARLY on this homework, as it might take some time to complete.

We do NOT expect you to parallelize python for loops in other ways (like using multithreading in python). We also do not expect you to use GPUs.



# Question 1: Supervised finetuning for a pretrained language model (40 pts)

In this question, you will be asked to finetune a pretrained language model DistillBERT on sentiment analysis task using IMDb dataset.

Each assertion worth 10 points. If the assertion doesn't pass, 10 points will be deducted. If the training loss does not decrease, 10 points will be deducted.

In [ ]:
!pip install datasets

In [ ]:
#load dataset
# we use imdb dataset to finetune the model. We test the model on imdb and sst2.
from datasets import load_dataset

imdb_dataset = load_dataset('stanfordnlp/imdb')


## 1.1. Preprocess the data (5pts)


For imdb, split the train dataset into train set and dev set. The dev set is used within training process to select the best checkpoint. Use train_test_split() from sklearn to split. The ratio of dev set is 0.1. Set the random state as 42.

In [ ]:
from sklearn.model_selection import train_test_split
# write your code here, make sure you use the name defined below.

#IMDb train split: texts and binary sentiment labels (0 neg, 1 pos)
_imdb_train = imdb_dataset["train"]
_texts = list(_imdb_train["text"])
_labels = list(_imdb_train["label"])

train_x, dev_x, train_y, dev_y = train_test_split(
    _texts, _labels, test_size=0.1, random_state=42, stratify=_labels
)


In [ ]:
print(train_x[0])

## 1.2. Prepare the data (10pts)


We use Dataset class from torch.utils.data to prepare data, and DataLoader class to prepare batches for training.

In the SentimentAnalysisDataset class, you need to use DistillBert tokenizer to tokenize the sentence.

In [ ]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')


In [ ]:
from torch.utils.data import Dataset, DataLoader

import torch


class SentimentAnalysisDataset(Dataset):
  def __init__(self,data, tokenizer, max_len = 512):
    self.input = data['input']
    self.input_ids = None
    self.attention_masks = None
    self.label = data['label']
    self.len = len(self.input)
    self.tokenizer = tokenizer

    self.max_len = max_len
    self.prepare()

  def prepare(self):
    input_ids = []
    attention_masks = []
    #write your code here
    for text in self.input:
        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        input_ids.append(encoded["input_ids"])
        attention_masks.append(encoded["attention_mask"])

    self.input_ids = torch.cat(input_ids, dim=0)
    self.attention_masks = torch.cat(attention_masks, dim=0)


  def __len__(self):
    return self.len

  def __getitem__(self,idx):
    return self.input_ids[idx], self.attention_masks[idx], self.label[idx]

# Example of usage
# Usage of GPU: due to limit usage of GPU on Colab, we will not train the whole training set. If you can get access to GPU, we strongly recommend you to run it on GPU and try it on the whole dataset. In this homework, we only run first 20 samples.
train = {'input':train_x[:20], 'label':train_y[:20]}
train_dataset = SentimentAnalysisDataset(train, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size = 4, shuffle = True)


In [ ]:
assert sum(train_dataset.attention_masks[0])==149

In [ ]:
dev = {'input': dev_x[:20], 'label': dev_y[:20]}
dev_dataset = SentimentAnalysisDataset(dev, tokenizer)

dev_dataloader = DataLoader(dev_dataset, batch_size = 4, shuffle = True)

## 1.3. Define the model (15pts)

We use DistillBERT as base model. We still need a linear layer to mapping the last hidden state to classes dimension.

The model should have a base model, a linear layer, a dropout layer (0.5) and softmax function. The forward function go through all the layers one by one and return the softmax result.

In [ ]:
from torch import nn
torch.manual_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
bertmodel = AutoModel.from_pretrained("distilbert/distilbert-base-uncased")

class ClassificationModel(nn.Module):
    def __init__(self,base_model,num_classes):
        super().__init__()
        #write your code here
        self.base_model = base_model
        self.dropout = nn.Dropout(0.5)
        hidden = base_model.config.hidden_size
        self.linear = nn.Linear(hidden, num_classes)

    def forward(self,input_ids, attention_mask):
        #write your code here
        #single-sequence test tensors are 1D; training batches are 2D
        if input_ids.dim() == 1:
          input_ids = input_ids.unsqueeze(0)
          attention_mask = attention_mask.unsqueeze(0)
        #DistilBERT: use [CLS] position (index 0) as sentence representation
        out = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)
        logits = self.linear(pooled)
        #return raw logits, nn.CrossEntropyLoss applies log-softmax internally
        return logits


In [ ]:
model = ClassificationModel(base_model = bertmodel, num_classes = 2 )

In [ ]:
#test case
input_ids = train_dataset.input_ids[0]
attention_mask = train_dataset.attention_masks[0]
model.eval()
with torch.no_grad():
  predicts = model(input_ids,attention_mask)


In [ ]:
predicts.tolist()

## 1.4. Model finetuning (10pts)

In this section, you need to implement the train and evaluation loops.

 Initialize the optimizer. We use AdamW for optimizer and cross entropy loss. (3pts)

In [ ]:
# write your code here
#move head + encoder to the same device in training loop
model = model.to(device)
#AdamW + cross-entropy is standard for multi-class classification on top of BERT
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
CELoss = nn.CrossEntropyLoss()


We use pytorch to implement. For each epoch, we run one training loop and one evaluation loop. At the end of training, we run the model on test set using the best model saved. For one training step, we run forward pass using pretrained model given input. Then we calculate loss and do backward propagation. See the instructions in the code block.  (7pts)

Ideally, you should see a obvious decrease in train loss, but no decrease in dev loss, since the model is overfitting on a small training set.


In [ ]:
from tqdm import tqdm
epochs = 10 #you can change this

best_loss = 10000
num_training_steps = epochs * len(train_dataloader)
best_model = None
with tqdm(total=num_training_steps, desc='Finetuning:') as pbar:
  for epoch in range(epochs):
    # training loop
    model.train()
    train_loss = 0
    for batch in train_dataloader:
      '''
      Tips:
      1. Put the input and model on the same device
      2. Use the optimizer correctly
      3. Update the train loss. The printed train loss should be train_loss/len(train_dataloader)
      '''
      optimizer.zero_grad()

      input_id, attention_mask, target = batch
      # Align batch tensors with model parameters (CPU or CUDA)
      input_id = input_id.to(device)
      attention_mask = attention_mask.to(device)
      target = target.to(device).long()

      outputs = model(input_ids = input_id, attention_mask = attention_mask)

      loss = CELoss(outputs,target)

      loss.backward()

      optimizer.step()

      train_loss += loss.item()

      pbar.update(1)

    print(f'Epoch {epoch}: train loss is {train_loss/len(train_dataloader)}')

    model.eval()
    with torch.no_grad():
      # Accumulate dev loss across all dev batches (reset once per epoch)
      dev_loss = 0
      for batch in dev_dataloader:
        '''
        Tips:
        1. You don't need to use optimizer
        2. Update the dev loss. The printed dev loss should be dev_loss/len(dev_dataloader)
        3. Save the checkpoint if the dev loss is smaller than best loss and update the best loss to dev loss
        '''
        input_id, attention_mask, target = batch
        input_id = input_id.to(device)
        attention_mask = attention_mask.to(device)
        target = target.to(device).long()

        outputs = model(input_ids = input_id, attention_mask = attention_mask)

        loss = CELoss(outputs,target)
        dev_loss += loss.item()

      avg_dev_loss = dev_loss / len(dev_dataloader)
      print(f'Epoch {epoch}: dev loss is {avg_dev_loss}')
      if avg_dev_loss < best_loss:
        #save checkpoint
        print(f'The best loss is {avg_dev_loss}. Saving checkpoint!')
        best_loss = avg_dev_loss
        best_model = model.state_dict()






# Question 2: Tokenization (25 pts)

How does one represent textual input to language models? One strategy that we have seen is to split up words on spaces, e.g.,

> This is an example.

> [This, is, an, example],

but this fails when unseen words appear at test time, e.g.,

> We named our son nwonkun.

> [We, named, our, son, \<unk\>] (5 tokens).

One solution to this problem is to use character-level tokens

> [W, e, _, n, a, m, e, d, _, o, u, r, _, s, o, n, _, n, w, o, n, k, u, n]

(24 tokens, if I counted right), but now the number of tokens required to encode a sentence has increased a *lot*.

## 2.1 Byte-pair encodings and sub-word tokenization (15 pts)

[Byte-pair encodings (BPE)](https://en.wikipedia.org/wiki/Byte_pair_encoding) are a clever middle ground for the tokenization problem.
Starting from a character-level tokenization, iteratively combine the most common bigrams (token pairs) into their own tokens.
For example, the most common bigrams from the previous example are "_n" and "on". Breaking the tie arbitrarily and creating a new token "_n" we now have

> [W, e, _n, a, m, e, d, _, o, u, r, _, s, o, n, _n, w, o, n, k, u, n]

reducing the token count to 22. Iteratively applying this rule, we can further reduce it to 20 tokens by adding the token "on", and so on. Each step of this algorithm greedily reduces the token count by the maximum amount.

This tokenization scheme, known as "sub-word tokenization" takes the best of both worlds: since the vocabulary still contains tokens for every byte, we never have to use the \<unk\> token, while still reducing the number of required tokens to encode a sequence. The more tokens you add, the shorter your sequence gets.

To decide which tokens to add to the vocabulary, we have to *train* our BPE tokenizer on a corpus.
In this section you will do just that.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
train: str = str.join(" ", dataset["train"]["text"])[:pow(10, 6)]
test: str = str.join(" ", dataset["test"]["text"])[:pow(10, 6)]

In [ ]:
from itertools import chain, pairwise
from collections import Counter
from tqdm import tqdm

class Tokenizer:
    # The lookup list contains *byte groups*, represented as a tuples of ints.
    # The token ID for a byte group is its index in the list.
    vocab: list[tuple[int, ...]]

    def __init__(self, training_seq: str, vocab_size: int) -> None:
        # Initialize a lookup with single-byte groups
        self.vocab = [(i,) for i in range(pow(2, 8))]
        # Working tokenization of the corpus: each byte is its own tuple token
        tokens: list[tuple[int, ...]] = [
            (b,) for b in bytes(training_seq, "utf-8")
        ]

        def merge_pair(seq: list[tuple[int, ...]], left: tuple, right: tuple):
            """Replace non-overlapping occurrences of (left, right) with left+right."""
            out: list[tuple[int, ...]] = []
            i = 0
            while i < len(seq):
                if i + 1 < len(seq) and seq[i] == left and seq[i + 1] == right:
                    out.append(left + right)
                    i += 2
                else:
                    out.append(seq[i])
                    i += 1
            return out

        for _ in tqdm(range(pow(2, 8), vocab_size)):
            """
            TODO: iteratively add the most common token pairs to the vocabulary.
            Advice: try using Counter and pairwise from the python std lib.
            """
            if len(tokens) < 2:
                break
            #count adjacent symbol pairs in the current tokenization
            pair_counts: Counter = Counter(pairwise(tokens))
            (best_left, best_right), _ = pair_counts.most_common(1)[0]
            new_tok = best_left + best_right
            self.vocab.append(new_tok)
            tokens = merge_pair(tokens, best_left, best_right)

        #longest-prefix
        self._vocab_by_len: list[tuple[int, tuple[int, ...]]] = sorted(
            enumerate(self.vocab), key=lambda x: len(x[1]), reverse=True
        )

    def tokenize(self, seq: str) -> list[int]:
        """
        TODO: convert a byte sequence into a token sequence by greedily adding
        the longest token that matches the rest of the sequence, e.g.,
        vocab = [a, aa, b]
        sequence = aaab
        token_seq = [1, 0, 2] NOT [0, 1, 2].
        """
        byte_seq: list[int] = list(bytes(seq, "utf-8"))
        ids: list[int] = []
        pos = 0
        while pos < len(byte_seq):
            matched = False
            #greedy longest match, check longest tuples first
            for vid, tup in self._vocab_by_len:
                L = len(tup)
                if pos + L <= len(byte_seq) and tuple(byte_seq[pos : pos + L]) == tup:
                    ids.append(vid)
                    pos += L
                    matched = True
                    break
            if not matched:
                #fallback: single-byte tokens
                ids.append(byte_seq[pos])
                pos += 1
        return ids

    def detokenize(self, token_seq: list[int]) -> str:
        # TODO: convert a token sequence into a byte sequence.
        byte_seq: list[int] = []
        for tid in token_seq:
            byte_seq.extend(self.vocab[tid])
        return bytes(byte_seq).decode("utf-8")


train_data = train[:10000]
tokenizer = Tokenizer(train_data, vocab_size=500)

print("Some of our new tokens:")
for token in tokenizer.vocab[-10:]:
    print(repr(bytes(token).decode("utf-8")))

As a sanity check, your implementation should be able to compress the training set to ~40-50% of its original size.
You should notice that the test set compression does not perform as well. This is because the distribution of bigrams in the test set does not exactly match the that of the train set. This gets worse the further your test set distribution is from your training set.

In [ ]:
# Do not edit this code cell
test_data = test[:10000]
train_bytes_len = len(bytes(train_data, "utf-8"))
train_token_len = len(tokenizer.tokenize(train_data))
print(f"Compressed train set to {train_token_len / train_bytes_len * 100:.0f}% original size")
test_bytes_len = len(bytes(test_data, "utf-8"))
test_token_len = len(tokenizer.tokenize(test_data))
print(f"Compressed test set to {test_token_len / test_bytes_len * 100:.0f}% original size")

assert train_data == tokenizer.detokenize(tokenizer.tokenize(train_data))
assert test_data == tokenizer.detokenize(tokenizer.tokenize(test_data))

## 2.2 BPE performance on OOD text. (5 pts)

Explore how English-trained BPE performs on non-English text by downloading corpora from a few different languages and using your English-trained tokenizer. What do you find? Do the results match your expectations? For what langauges does the tokenizer struggle with the most? How might this impact society if everyone were to use your tokenizer?

Include your code, results, and discussion in new cells below.

Hint: we recommend you use `load_dataset` to fetch from HuggingFace with `streaming=True` to avoid huge downloads. You might want to take a look at the `oscar` dataset.

In [ ]:
#compare English-trained BPE on public multilingual text
from datasets import load_dataset

def compression_ratio(text: str, tok: Tokenizer) -> float:
    b = len(text.encode("utf-8"))
    return float("nan") if b == 0 else len(tok.tokenize(text)) / b

#english baseline from the corpus used to train the tokenizer in 2.1
en_text = train[:8000]

#public OPUS-100 language pairs; grab the non-English side of each translation pair
lang_configs = {
    "es": ("en-es", "es"),
    "zh": ("en-zh", "zh"),
}

samples = {}
for lang, (config, target_lang) in lang_configs.items():
    try:
        ds = load_dataset("opus100", config, split="train", streaming=True)
        parts = []
        for row in ds:
            text = row["translation"][target_lang]
            if len(text) >= 50:
                parts.append(text)
            if len(parts) >= 100:
                break
        samples[lang] = "\n".join(parts)[:8000]
    except Exception as e:
        print(f"Skipping {lang}: {e}")

#so the cell still runs if HF is unavailable
if not samples:
    samples = {
        "es": "El rápido zorro marrón salta sobre el perro perezoso. " * 30,
        "zh": "快速的棕色狐狸跳过懒狗。" * 30,
        "ar": "الثعلب البني السريع يقفز فوق الكلب الكسول. " * 30,
    }

print("Token length / byte length (lower = better compression):")
print(f"  en (wikitext slice): {compression_ratio(en_text, tokenizer):.3f}")
for lang, text in samples.items():
    print(f"  {lang}:                 {compression_ratio(text, tokenizer):.3f}")

QUESTIONS: Using an English-trained BPE tokenizer, there are obvious differences. English compresses the best at 0.42, Spanish is a little bit worse at 0.64, and Chinese is the worst at 0.98.

This is expected because the tokenizer was trained on English, so it captures English patterns well and because Spanish has a similar alphabet, it performs reasonably, while Chinese has a completely different writing system, leading to poor compression.

The tokenizer struggles most with Chinese and other non-Latin scripts. If everyone used this tokenizer, non-English languages would be less efficiently represented, increasing costs and reducing model quality, which could reinforce language inequality in AI systems.

## 2.3 Pitfalls of and alternatives to BPE (5 pts)

BPE tokenization sufferes from other issues as well. Due to the implementation of our BPE tokenizer, detokenizing a sequence of tokens then re-tokenizing it does not always recover the original sequence:
```
vocab = {a, aa, b}
tokens = [0, 1, 2]
detokenized = aaab
retokenized = [1, 0, 2]
```

Another issue is that some tokens that may have been prevalent during BPE training may not be present during language model training, leading to funky situations where the language model has not been trained to represent or output some tokens. See this paper for more information: https://arxiv.org/pdf/2405.05417.

Some NLP researchers think that we should move away from sub-word tokenization to get rid of these problems. Engage with this discussion by either
- Finding a paper that points out an issue with tokenization and propose your own solution for how you would fix it, or
- Finding a paper that proposes an alternative tokenization scheme (or way of processing text) and discuss the drawbacks of the proposed method.

Your response should be about a paragraph in length and link to a paper.

ALTERNATIVE: One alternative to BPE is proposed in the T-FREE: Tokenizer-Free Generative LLMs via Sparse Representations paper, which removes tokenization entirely and represents text at the character level using sparse vectors. Instead of compressing text into subwords, each character is mapped to a high-dimensional vector with mostly zero values, allowing the model to process all languages uniformly without relying on a predefined vocabulary. This directly addresses the issue observed in part 2.2, where an English-trained tokenizer performed poorly on Spanish and Chinese due to the lack of shared subword structure. However, the drawbacks ot this are because it operates at the character level, input sequences become significantly longer, increasing both memory usage and computational cost during training and inference. Also, unkike BPE, it does not provide meaningful subword units, making it harder for the model to learn semantic patterns efficiently. While T-FREE improves language fairness and avoids tokenizer bias, it sacrifices efficiency and structured representation, which can make it less practical in real-world systems. https://arxiv.org/html/2406.19223v1


# Question 3: Generation Algorithms (35 pts)

In this problem, we will implement several common decoding algorithms and test them with the GPT-2 Medium model.

Given the class below, we will fill in each of the method stubs. You may create additional helper methods as well to make components re-usable.

**You are not allowed to use the generate() function in the transformers library. You can only use the model's forward() method to retrieve final layer logits**

In addition to the methods we ask you to implement, which are:
- Greedy decoding
- Temperature Sampling
- Nucleus Sampling

You will choose ONE of the following sampling algorithms to implement as well (make sure to add your own method, since we do not provide one by default):
- Typical Sampling ([Meister et al. (2022)](https://arxiv.org/abs/2202.00666))
- Eta Sampling ([Hewitt et al. (2022)](https://arxiv.org/abs/2210.15191))

Points for this question will be distributed as follows:

- 5-10 points for implementing each decoding algorithm
- 5 points for implementing the generate() function (you will make this incrementally through each sub-part)
- 5 points for filling out the table with list of tokens (see instructions below)

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import Optional

class LM():
  def __init__(self, model_name: str = "openai-community/gpt2-medium"):
    self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    self.model = AutoModelForCausalLM.from_pretrained(model_name)
    self.model.eval()
    # GPT-2 has no pad token by default; reuse EOS for padding-safe batching
    if self.tokenizer.pad_token is None:
      self.tokenizer.pad_token = self.tokenizer.eos_token

  def _device(self):
    return next(self.model.parameters()).device

  def _encode_prompt(self, prompt: str) -> torch.Tensor:
    ids = self.tokenizer.encode(prompt, return_tensors="pt")
    return ids.to(self._device())

  def greedy_decoding(self, prompt: str, max_length: int = 64) -> str:
    """
    TODO:

    Implement greedy decoding, in which we use the highest
    probability token at each decoding step
    """
    input_ids = self._encode_prompt(prompt)
    for _ in range(max_length - input_ids.shape[1]):
      with torch.no_grad():
        logits = self.model(input_ids=input_ids).logits[:, -1, :]
      next_id = torch.argmax(logits, dim=-1, keepdim=True)
      input_ids = torch.cat([input_ids, next_id], dim=1)
      if next_id.item() == self.tokenizer.eos_token_id:
        break
    return self.tokenizer.decode(input_ids[0], skip_special_tokens=True)

  def temperature_sampling(self, prompt: str, temperature: float = 1.0, max_length: int = 64) -> str:
    """
    TODO:

    Implement temperature sampling, in which we sample
    from the output distribution at each decoding step,
    with a temperature parameter to control the "peakiness"
    of the output distribution
    """
    #T=0 degenerates to greedy
    if temperature <= 1e-5:
      return self.greedy_decoding(prompt, max_length=max_length)
    input_ids = self._encode_prompt(prompt)
    for _ in range(max_length - input_ids.shape[1]):
      with torch.no_grad():
        logits = self.model(input_ids=input_ids).logits[:, -1, :]
      logits = logits / temperature
      probs = F.softmax(logits, dim=-1)
      next_id = torch.multinomial(probs, num_samples=1)
      input_ids = torch.cat([input_ids, next_id], dim=1)
      if next_id.item() == self.tokenizer.eos_token_id:
        break
    return self.tokenizer.decode(input_ids[0], skip_special_tokens=True)

  def nucleus_sampling(self, prompt: str, p: float = 0.9, max_length: int = 64, temperature: float = 1.0) -> str:
    """
    TODO:
    Implement nucleus sampling, in which we
    sample from a subset of the vocabulary
    at each decoding step
    Note: There is also a temperature parameter here
    """
    input_ids = self._encode_prompt(prompt)
    for _ in range(max_length - input_ids.shape[1]):
      with torch.no_grad():
        logits = self.model(input_ids=input_ids).logits[:, -1, :]
      logits = logits / max(temperature, 1e-8)
      probs = F.softmax(logits, dim=-1)
      sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
      cumsum = torch.cumsum(sorted_probs, dim=-1)
      #HuggingFace-style nucleus
      sorted_remove = cumsum > p
      sorted_remove[..., 1:] = sorted_remove[..., :-1].clone()
      sorted_remove[..., 0] = False
      filtered_sorted = sorted_probs.masked_fill(sorted_remove, 0.0)
      filtered_sorted = filtered_sorted / filtered_sorted.sum(dim=-1, keepdim=True)
      choice = torch.multinomial(filtered_sorted, num_samples=1)
      next_id = sorted_idx.gather(-1, choice)
      input_ids = torch.cat([input_ids, next_id], dim=1)
      if next_id.item() == self.tokenizer.eos_token_id:
        break
    return self.tokenizer.decode(input_ids[0], skip_special_tokens=True)

  def typical_sampling(
      self,
      prompt: str,
      mass: float = 0.9,
      temperature: float = 1.0,
      max_length: int = 64,
  ) -> str:
    """Locally typical sampling (Meister et al., 2022): sample among tokens whose -log p is near entropy."""
    input_ids = self._encode_prompt(prompt)
    for _ in range(max_length - input_ids.shape[1]):
      with torch.no_grad():
        logits = self.model(input_ids=input_ids).logits[:, -1, :]
      logits = logits / max(temperature, 1e-8)
      probs = F.softmax(logits, dim=-1)
      entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1, keepdim=True)
      nll = -torch.log(probs + 1e-10)
      deviation = torch.abs(nll - entropy)
      sorted_dev, sort_idx = torch.sort(deviation, dim=-1)
      sorted_p = probs.gather(-1, sort_idx)
      cumsum = torch.cumsum(sorted_p, dim=-1)
      sorted_remove = cumsum > mass
      sorted_remove[..., 1:] = sorted_remove[..., :-1].clone()
      sorted_remove[..., 0] = False
      filtered = sorted_p.masked_fill(sorted_remove, 0.0)
      filtered = filtered / filtered.sum(dim=-1, keepdim=True)
      pick = torch.multinomial(filtered, num_samples=1)
      next_id = sort_idx.gather(-1, pick)
      input_ids = torch.cat([input_ids, next_id], dim=1)
      if next_id.item() == self.tokenizer.eos_token_id:
        break
    return self.tokenizer.decode(input_ids[0], skip_special_tokens=True)

  def generate(
      self,
      prompt: str,
      temperature: float = 1.0,
      p: Optional[float] = None,
      use_typical: bool = False,
  ) -> str:
      """
      TODO:

      Route to the appropriate generation function
      based on the arguments
      HINT: What parameter values should map to greedy decoding?
      """
      #temperature==0 recovers deterministic argmax (greedy) at each step
      if use_typical:
        return self.typical_sampling(prompt, mass=0.9, temperature=temperature)
      if temperature == 0:
        return self.greedy_decoding(prompt)
      if p is not None:
        return self.nucleus_sampling(prompt, p=p, temperature=temperature)
      return self.temperature_sampling(prompt, temperature=temperature)

In [ ]:
GPT2LM = LM("openai-community/gpt2-medium")

For each sampling algorithm you implement, fill out this table, in which you will list the top 10 highest probability tokens **at the first decoding step** in a comma separated list. For algorithms like nucleus sampling where you perform some kind of truncation/re-distribution of the output distribution, do the truncation/re-distribution first, and then sort the vocabulary by probability to complete the table.

For this and all questions below, use the following prompt:


**"Once upon a time in a land far far away, "**

Note: Use the default value for `max_length` for all questions below.

In [ ]:
import torch.nn.functional as F

PROMPT = "Once upon a time in a land far far away, "
GPT2LM.model.eval()
with torch.no_grad():
    ids = GPT2LM._encode_prompt(PROMPT)
    logits = GPT2LM.model(input_ids=ids).logits[:, -1, :]
    probs = F.softmax(logits, dim=-1)[0]

def show_top10(title, pvec):
    topk = torch.topk(pvec, k=10)
    tokens = []
    for i in topk.indices.tolist():
        piece = GPT2LM.tokenizer.convert_ids_to_tokens(i)
        #Ġ = leading space, Ċ = newline in GPT-2 vocab
        readable = piece.replace('Ġ', ' ').replace('Ċ', '\n')
        tokens.append(readable)
    print(title)
    print(", ".join(tokens))
    print()

show_top10("Greedy / Temperature (T=1.0)", probs)

#Nucleus (p=0.9)
p_cut = 0.9
sorted_probs, sorted_idx = torch.sort(probs, descending=True)
cumsum = torch.cumsum(sorted_probs, dim=0)
sorted_remove = cumsum > p_cut
sorted_remove[1:] = sorted_remove[:-1].clone()
sorted_remove[0] = False
nucleus_sorted = sorted_probs.masked_fill(sorted_remove, 0.0)
nucleus_sorted = nucleus_sorted / nucleus_sorted.sum()
nucleus = torch.zeros_like(probs)
nucleus.scatter_(0, sorted_idx, nucleus_sorted)
show_top10("Nucleus (p=0.9) after truncate + renormalize", nucleus)

#Typical (mass=0.9)
entropy = -(probs * torch.log(probs + 1e-10)).sum()
nll = -torch.log(probs + 1e-10)
deviation = torch.abs(nll - entropy)
sorted_dev, sort_idx = torch.sort(deviation)
sorted_p = probs[sort_idx]
cumsum2 = torch.cumsum(sorted_p, dim=0)
sorted_remove2 = cumsum2 > 0.9
sorted_remove2[1:] = sorted_remove2[:-1].clone()
sorted_remove2[0] = False
typ_sorted = sorted_p.masked_fill(sorted_remove2, 0.0)
typ_sorted = typ_sorted / typ_sorted.sum()
typ = torch.zeros_like(probs)
typ.scatter_(0, sort_idx, typ_sorted)
show_top10("Typical sampling (mass=0.9) after truncate + renormalize", typ)


| **Decoding Algorithm** | **10 Highest Probability Tokens** |
|------------------------|-----------------------------------|
| Greedy                 | there, once, long, ich, 【, ~~, 〒, 0, far, the |
| Temperature (t=1.0)    | there, once, long, ich, 【, ~~, 〒, 0, far, the |
| Nucleus (p=0.9)        | there, once, long, ich, 【, ~~, 〒, 0, far, the |
| Typical/Eta                | there, once, long, ich, 【, ~~, 〒, 0, far, the|

## 3.1 Greedy Decoding (5 points)

First, implement the most simple decoding method of greedy decoding. Here, at each decoding time step, simply use the highest probability token. Note that you'll need to adjust the generate function so that a specific temperature value will map to greedy decoding (what should that value be?).

Use the prompt given above to test your implementation. What do you notice?

I noticed that Greedy decoding always picks the single highest probability token at each step, which corresponds to temperature=0 in the generate() function. When tested on the prompt above, greedy decoding tends to produce text that is very deterministic and repetitive. Running it multiple times gives the same continuation, and the output often becomes less creative and more generic because the model always chooses the locally most likely next token at each step.

## 3.2 Temperature Sampling (10 pts)

Sometimes (a lot of the time?), we don't actually just want the highest probability token at each time step. Why might this be the case?

Greedy decoding always picks the most likely token, which leads to repetitive, generic, and less creative text. The model gets trapped in high-probability loops and never explores the other coherent continuations that have slightly lower probability at individual steps but ultimately would produce more natural and diverse text overall. Sampling introduces randomness that better reflects the true distribution of plausible continuations.

To adjust for this, we often use sampling algorithms instead of greedy decoding. However, there are many ways we can go about sampling.

First, implement temperature sampling. Recall that the temperature parameter adjusts the "randomness" of the output at each time step. Here, you'll need to think about how to adjust the output distribution which you will do multinomial sampling from. Be careful about how you will handle very low (close to 0) temperatures.

Given the same prompt as above, test your implementation with the following temperature values: [0.3, 0.5, 0.7, 0.9, 1.1]. For each value, sample 3 outputs. What do you notice in terms of the differences between output sets across different temperature values?  

In [ ]:
PROMPT = "Once upon a time in a land far far away, "
torch.manual_seed(0)
for T in [0.3, 0.5, 0.7, 0.9, 1.1]:
    print(f"\n=== temperature = {T} ===")
    for i in range(3):
        out = GPT2LM.temperature_sampling(PROMPT, temperature=T, max_length=64)
        print(f"  sample {i+1}: {out!r}")


| **Temperature** | **Output 1** | **Output 2** | **Output 3** |
|-----------------|--------------|--------------|--------------|
| 0.3             | 'Once upon a time in a land far far away, \xa0I was a young boy. I was a child of the desert. I was a child of the desert. I was a child of the desert. I was a child of the desert. I was a child of the desert. I was a child of' | 'Once upon a time in a land far far away, \xa0a young man was born. \xa0The name of that young man was \xa0Ruthless. \xa0Ruthless was the name of the god of the land. \xa0Ruthless was the name of the god of the land.' | 'Once upon a time in a land far far away, \xa0I was a child. I was a child of the forest. I was a child of the woods. I was a child of the forest. I was a child of the woods. I was a child of the woods. I was a child of the'
 |
| 0.5             | 'Once upon a time in a land far far away, \xa0a young girl named \xa0Kara was caught in a war between two powerful kingdoms. \xa0The war was so fierce that it was said that every princess and king on the planet would fall. \xa0Kara was rescued by a mysterious boy,'
 | 'Once upon a time in a land far far away, \xa0there was a great king, and he was very wise. He knew how to make men of all kinds, and he knew the use of all kinds of weapons. He also knew how to make men lovely, and he loved beautiful women. And he loved'
 | 'Once upon a time in a land far far away, \xa0a young boy with a bright future was born.\nHis name was\xa0 Malak .\nHe was the son of a\xa0 famous\xa0 father \xa0who lived in the\xa0 north .\nAnd his mother was a beautiful woman,\xa0 the daughter of'
 |
| 0.7             | 'Once upon a time in a land far far away, \xa0you were a girl with a brother, who was an orphan from a foreign land. \xa0One day, you were abducted by a mysterious woman, who took you to a temple of the holy god. \xa0You were placed into a strange land that'
 | 'Once upon a time in a land far far away, \xa0a young man was born the doom of everything he had ever done. \xa0This man fought his way to the top of a great chain in the sky, like a star-folded dove which is ready to fly into the sea, but without wings'
 | 'Once upon a time in a land far far away, _______ was the center of the universe, and _______ was the world. Now that it is not _______, it is _______ , _______ , and _______ , and _______ . Since everyone knows, everyone knows, everyone knows, that _______'
 |
| 0.9             | "Once upon a time in a land far far away, \xa0it was lonely, lonely as anysharing a bed for the first time. The p.m. at the ping-pong table wasn't bad either. There were several girls at the table, and no boys. The boys stayed at home till "
 | 'Once upon a time in a land far far away, \xa0a high hill stood overlooking the plains. \xa0Near by there lived two men,\xa0\na young man and a tall man. \xa0While in the area one day, \xa0the tall man passed this hill and looked to be in good health'
 | 'Once upon a time in a land far far away, ursinia came unto humankind to be one of us. When we became aware of its existence, we sent out legends, as is their wont, of its terrible beings. They speak of its power, of an enemy, of its pinnacle, and the shape of'
 |
| 1.1             | 'Once upon a time in a land far far away, \xa0I dreamed I smelled horn blossom and white cake. \xa0I imagined concerts engulfed in thick tobacco and glistening wreaths of brandy, courts of white, slaves ordained to bless the battlefield shoulders and maternal satyrs making travelers fill their flesh'
 | "Once upon a time in a land far far away, \xa0there was a TI League team called Sorac as well named Shanxi Blue. Here's your first generation of the hidden cosmetic in Big final showdown:What would you actually be excited to see from them.Ether's Bringing On Anime Still. Watching players"
 | 'Once upon a time in a land far far away, \ue040 Helen of Troy approached mythology with Spartan intention. Without a verse memorial to towns and fields and streets upon hills, while amidst the clouds above provided, causes and mankind, as well as things that joy and pity and lament, soared tears at whose'
 |

At low temperature T=0.3, the outputs are highly repetitive and formulaic as seen in sample 1 where it repeats "I was a child of the desert" multiple times, showing the model getting stuck in a loop. At T=0.5, the outputs become more coherent and varied, producing distinct story beginnings without as much repetition. At T=0.7, the three samples diverge more noticeably from each other, with different characters and settings emerging across samples. At T=0.9, the outputs become more imaginative and unpredictable, with more unusual ideas and less overlap between the three samples. At T=1.1, the outputs are the most diverse and creative but also the least coherent, producing combinations like "Helen of Troy approached mythology with Spartan intention" that don't make much sense. Overall, lower temperatures produce more consistent but repetitive text, while higher temperatures produce more creative and diverse outputs at the cost of coherence. Somewhere in the middle, is the spot where you can have somewhat creative text but maintain consistency and natural flow.

## 3.3 Nucleus Sampling (10 pts)

Originally published in [Holtzmann et al. (2021)](https://arxiv.org/abs/1904.09751), nucleus sampling was designed to address an issue that was especially prevalent in language models at the time.

This issue is the case of "neural text degeneration," where outputs from LMs would often degenerate into gibberish if a low probability token was ever decoded. To address this, nucleus (also known as top-p) sampling uses a hyperparameter, p, to control how big of a subset of the vocabulary we sample from at each step. For example, if p=0.9, we only sample from the subset of tokens that have a cumulative probability mass of 0.9 (after sorting by probability).

Implement nucleus sampling and then use the same prompt as above and test your implementation with the following p-values: [0.97, 0.95, 0.9, 0.8, 0.7]
What do you notice across outputs?

In [ ]:
PROMPT = "Once upon a time in a land far far away, "
torch.manual_seed(1)
for p in [0.99, 0.95, 0.9, 0.8, 0.5]:
    print(f"\n=== top-p = {p} ===")
    for i in range(3):
        out = GPT2LM.nucleus_sampling(PROMPT, p=p, max_length=64, temperature=1.0)
        print(f"  sample {i+1}: {out!r}")


| **p** | **Output 1** | **Output 2** | **Output 3** |
|----------|--------------|--------------|--------------|
| 0.99     | 'Once upon a time in a land far far away, \xa0I came upon a truth. Well known among the common folk of virtuality were facts: \xa0the art and science of baking. \xa0I believed that strong primary colors were the rich source and pleasure in the flavor of baked goods. Cookies enhanced this'
 | "Once upon a time in a land far far away, \xa0a boy and girl met at a pub in twilight, when he groped her behind her back and her back ripped apart at the force of his thrust and on the control of her arms, Samie stretched her arms around her husband Klaatu's other"
 | "Once upon a time in a land far far away, \xa0Alyona\xa0in her glory from life and love wouldn't be . . . \xa0She would be. She would have been that I think every mortal would want to be. e If all the passengers in an aircraft are stolen from the grave ."
 |
| 0.95     | 'Once upon a time in a land far far away, ------------ Akidaur------ was seen talking to the camera. It was an image only heard once, before starting off into discussion with Donald. "(...my daughter\'s wearing a room key after all.)" --- ..... ...t...T. ....... ..'
 | 'Once upon a time in a land far far away, \xa0there lived a man named Moses. \xa0He was without friends, he was alone and he thirsted. \xa0Moses, hoping to meet him, visited a village called Sidon. \xa0\xa0\xa0 He saw alone, dreamt\xa0quothe wide'
 | "Once upon a time in a land far far away, _____... > > Folks, I'm head mover instead of follows in King Pink's wake. Though I'll try my best to keep up with you all! > > So, just who are you, exactly? > > I'm @scbb"
 |
| 0.9      | 'Once upon a time in a land far far away, ikebe heroes and nienos fought and riled up a frenzy among those nations ikebe, insane nienos, and iranians . They caused innumerable riots and strife . ikebe ukazar & iran started to fight and'
 | 'Once upon a time in a land far far away, \xa0Lolita\xa0performed\xa0an awesome rendition of\xa0Bob Dylan\'s\xa0"The Song Remains the Same" as an instrumental during San Antonio City Hall\'s\xa0Civic Hall Entrance Main Street Acoustics. Lolita did it with blue'
 | 'Once upon a time in a land far far away, ㅠㅠ I was afraid I had a bad fever. ㅠㅠ "The cure I will give you is ㅠ" You are my healer ㅠㅠ Even though my eyes are big'
 |
| 0.8      | "Once upon a time in a land far far away, \xa0the race of man is strong and silent.\xa0 The hero's battle is his victory.\xa0 He is indeed strong.\xa0 But it is his defeat that makes him powerful.\xa0 The hero's sacrifice is the redeeming effect of that strength.\xa0 To"
 | 'Once upon a time in a land far far away, \xa0the monk born into the sin-stricken council did something that should have been a mark of mercy for the entire world, an act that should have brought salvation. The monk, in his own way, was a victim of greed. It was his greed'
 | "Once upon a time in a land far far away, 《Legend》was written about the Gold of the Gold Mountain. What was written was written over the glory of the people. When the people came to hear it, they wouldn't believe it was true. Even now, 《Legend》"
 |
| 0.5      | 'Once upon a time in a land far far away, \xa0there lived a wise man named\xa0 Jiraiya \xa0who taught the wise man wisdom. And the wise man said to his disciples, "I want to know what you want to know. What do you want to know?"\xa0 "You want'
 | 'Once upon a time in a land far far away, _________ met ______. They became fast friends. _________ was a good person. ______ was a good person. _________ had his reasons for being, but they were always the same. He had been trying to get out of his job for a'
 | "Once upon a time in a land far far away, \xa0a man of noble birth and wealth, his name was Samba (God), and his father was Lord Shatua (Shit). \xa0Shatua's firstborn son was Samba's secondborn son, and the last son was Sam"
 |

At large p values like 0.99, the nucleus is very wide and includes most of the vocabulary, so outputs are highly diverse and sometimes incoherent, similar to pure sampling. As p decreases to 0.9 and 0.8, the nucleus shrinks to only the most probable tokens, producing more focused and coherent continuations while still maintaining some variety across the 3 samples. At p=0.5, the nucleus is very tight, keeping only the top tokens that together make up 50% of the probability mass, resulting in more repetitive outputs similar to low-temperature sampling. Overall, smaller p values reduce the risk of "neural text degeneration" by preventing the model from ever sampling a very low probability token, trading diversity for safety and coherence.

## 3.4 More variations on decoding algorithms (10 pts)

Nucleus sampling was definitely not the end of the road in terms of new decoding algorithms. Even in the past few years, new decoding algorithms have been proposed to address some limitations of existing algorithms.

Two in particular are:
- Typical Sampling ([Meister et al. (2022)](https://arxiv.org/abs/2202.00666))
- Eta Sampling ([Hewitt et al. (2022)](https://arxiv.org/abs/2210.15191))

For this question, CHOOSE ONE of the two algorithms presented above. Below, please describe in a few sentences what your chosen algorithm does in a novel way and the broad motivation behind it. Along with this description, present 3 sampled outputs for the same prompt as above (you can use one hyperparameter value for all of these).



I implement Typical Sampling (Meister et al., 2022), which selects tokens whose surprise is closest to the model’s average uncertainty (entropy). Instead of always choosing the most likely tokens like nucleus sampling, it favors tokens that are neither too predictable nor too unlikely. This better reflects natural human language as we tend to gravitate towards producing typical outputs that are not very surprising but also not predictable, and helps avoid the repetition of greedy decoding and the randomness of full sampling.

In [ ]:
#typical sampling (Meister et al., 2022): three samples, one hyperparameter setting
PROMPT = "Once upon a time in a land far far away, "
torch.manual_seed(2)
for i in range(3):
    out = GPT2LM.typical_sampling(PROMPT, mass=0.9, temperature=1.0, max_length=64)
    print(f"sample {i+1}: {out!r}")


| Decoding Method | **Output 1** | **Output 2** | **Output 3** |
|-----------------|--------------|--------------|--------------|
| Typical sampling (mass=0.9) |'Once upon a time in a land far far away, \xa0not many humanoid race traveled in the open air. Yet\xa0there was\xa0one noble people who carried out a daring raid against monster attacks. But that raid was made illegal due to suspicion of responsibility. Many strayed from the path and later the orders to'
 |'Once upon a time in a land far far away, \xa0I had been staring off into space, on a beautiful orange sky with the brightest stars in the sky, staring into emptiness, only to find myself staring back. Those familiar faces I saw when I looked up could very well not have been sighted from there'
 |'Once upon a time in a land far far away, \xa0I was quietly tossing random pieces of poop in an open mailbox.\nOccasionally I would catch myself staring at the piece of poop until the place I was talking about turned pink, or I would send out an\xa0 e-mail \xa0posting\xa0'
 |

# Question 4. Prompting (50 pts)

In this problem, we will try various prompting approaches and prompt an LLM for a Math Reasoning Benchmark called [GSM8K](https://github.com/openai/grade-school-math), which contains grade school math word problems. This is a very common _reasoning_ benchmark used to test various LLMs.

The LLM that we will be using is [Google Gemini](https://gemini.google.com/). We will be prompting Gemini by using an API call to the Gemini Model. Normally, you can also prompt Open Source LLMs via the HuggingFace Library, however due to compute constraints, we use Gemini in this problem.

## 4.0 Setting up the GSM8K Dataset and Google Gemini

Follow the steps below to download the GSM8K Dataset and to setup Google Gemini on Colab. You will automatically get points for this subpr

In [ ]:
from datasets import load_dataset

dataset = load_dataset("gsm8k", 'main')

In [ ]:
len(dataset['train']), len(dataset['test'])

In [ ]:
# An example instance of this dataset

dataset['test'][6]

### Gemini Setup (from the official [Gemini documentation](https://colab.research.google.com/github/google/generative-ai-docs/blob/main/site/en/gemini-api/docs/get-started/python.ipynb))


Before you can use the Gemini API, you must first obtain an API key. If you don't already have one, create a key with one click in Google AI Studio.

<a class="button button-primary" href="https://makersuite.google.com/app/apikey" target="_blank" rel="noopener noreferrer">Get an API key</a>

In Colab, add the key to the secrets manager under the "🔑" in the left panel.

---

Give it the name `GEMINI_API_KEY`.

Once you have the API key, pass it to the SDK. You can do this in two ways:

* Put the key in the `GEMINI_API_KEY` environment variable (the SDK will automatically pick it up from there).
* Pass the key to `genai.configure(api_key=...)`

In [ ]:
# All imports for this question
from google.colab import userdata
import google.generativeai as genai
from datasets import Dataset
import random
from typing import Callable, List, Any

In [ ]:
GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
# Test if your setup is working, do not change the model name
model = genai.GenerativeModel("gemini-2.5-flash-lite")
response = model.generate_content("What is Natural Language Processing? Explain it to a five year old.")
print(response.text)

## 4.1 Data and Prompting Setup (15 + 5 pts)

In this part, we will create some boilerplate code to process our dataset and generate prompts from the dataset.



### Processing the GSM8K Dataset

In [ ]:
def process_gsm8k_answers(dataset: Dataset) -> Dataset:
    """
    Processes the GSM8K dataset to remove reasoning chains and retain only the numerical answers.
    Assumes answers are separated from reasoning by the '###' string.

    Args:
    dataset (Dataset): Huggingface Dataset object for GSM8K.

    Returns:
    Dataset: Processed Dataset object with numerical answers only.
    """

    def extract_answer(sample):
        # IMPLEMENT HERE
        #split the answer using '####' and return a dictionary with the key 'processed_answer'
        raw = sample["answer"]
        #GSM8K stores "rationale #### final_number" so keep only the final numeric answer
        if "####" in raw:
            processed = raw.split("####")[-1].strip()
        else:
            processed = raw.strip()
        return {"processed_answer": processed}

    return dataset.map(extract_answer)

### Building Prompts (15 pts)

We will be implementing FIVE (5) prompting methods. See their descriptions below -
1. **Zero-Shot Answer Only (2 pts)**: You prompt the model to only generate the answer to the question

2. **Zero-Shot Chain of Thought (CoT) (3 pts)**: Refer to the [Chain of Thought Paper](https://arxiv.org/abs/2201.11903). CoT refers to a reasoning chain that is generated by the model before generating the actual answer. This has shown to improve performance. In this setup, you will prompt the model to generate a reasoning chain before the answer.

3. **5-Shot Answer Only (2 pts)**: You provide some in-context examples to prompt the model with to generate the answer. This is analogous to Approach 1. Use the a random set of 5 examples from the training set to create the in-context examples.

4. **5-Shot CoT (3 pts)**: Combine Approaches 2 and 3 to do 5-shot CoT prompting.

5. **Your own prompt! (5 pts)**: Try something new. Think about how you solve Math problems and implement your own prompting method.

In [ ]:
def prompt_generation_zero_shot(problem: str) -> str:
    """
    Zero-shot prompt.

    Returns:
    str: The generated prompt.
    """
    # IMPLEMENT HERE
    return (
        "Answer the following math problem. Output only the final numerical answer, with no explanation.\n\n"
        f"Problem: {problem}\n"
        "Answer:"
    )

In [ ]:
def prompt_generation_zero_shot_cot(problem: str) -> str:
    """
    Zero-shot Chain of Thought (CoT) prompt.

    Returns:
    str: The generated prompt.
    """
    # IMPLEMENT HERE
    return (
        "Solve the following math problem step by step. "
        "At the end, write your final answer on a new line starting with '####'.\n\n"
        f"Problem: {problem}\n"
        "Solution:"
    )

In [ ]:
def prompt_generation_5_shot(problem: str, training_set: Dataset) -> str:
    """
    5-shot prompt generation for GSM8K problems. Randomly selects 5 examples from the training set.

    Returns:
    str: The generated prompt with 5 in-context_examples.
    """
    # IMPLEMENT HERE
    random.seed(42)
    n = len(training_set)
    picks = random.sample(range(n), min(5, n))
    blocks = []
    for j in picks:
        row = training_set[j]
        ans = row.get("processed_answer")
        if ans is None:
            ans = row["answer"].split("###")[-1].strip()
        blocks.append(f"Problem: {row['question']}\nAnswer: {ans}")
    demos = "\n\n".join(blocks)
    return demos + f"\n\nProblem: {problem}\nAnswer:"


In [ ]:
def prompt_generation_5_shot_cot(problem: str, training_set: Dataset) -> str:
    """
    5-shot Chain of Thought (CoT) prompt generation. Randomly selects 5 examples
    from the training set and includes reasoning steps.

    Returns:
    str: The generated prompt with 5 CoT in-context examples.
    """
    # IMPLEMENT HERE
    random.seed(42)
    n = len(training_set)
    picks = random.sample(range(n), min(5, n))
    blocks = []
    for j in picks:
        row = training_set[j]
        blocks.append(f"Problem: {row['question']}\nSolution:\n{row['answer']}")
    demos = "\n\n".join(blocks)
    return demos + f"\n\nProblem: {problem}\nSolution:\n"


In [ ]:
# Feel free to change the method definition

def my_prompt(problem: str) -> str:
    """
    Your own unique way of prompting an LLM for Math word problems.

    Returns:
    str: The generated prompt
    """
    # IMPLEMENT HERE
    #its an expert tutor, map out question, prep answer, give final answer: very directional, strict
    return (
        "You are an expert math tutor. Solve the problem by:\n"
        "1. Identifying the key numbers and what is being asked.\n"
        "2. Writing out each calculation step clearly.\n"
        "3. Ending with exactly: Final Answer: <number>\n\n"
        f"Problem: {problem}\n"
    )

## 4.2 Prompting Gemini and Implementing Self-Consistency (5 + 5 + 10 pts)

Here, you will help build the wrapper for prompting Gemini using the prompt methods you have designed above.

You will then also implement Self-Consistency based prompting. Refer to the [Self-Consistency Paper](https://arxiv.org/abs/2203.11171). In order to implement Self-Consistency, you generate multiple Zero-Shot CoT (Approach 2 in the prompting methods) candidates, and take a majority vote of the answers predicted by each candidate.

### First, write the function where you will process the answer generated by the model. (5 pts)

Note that answer processing changes for different prompt types, so this function also takes in the name of the method in its argument.

In [ ]:
def answer_processing(prediction: str, prompt_function: Any) -> str:
    """
    Processes the model's generated output to extract the final answer.

    Returns:
    str: The processed numerical answer.
    """
    import re
    import time

    prompt_name = prompt_function.__name__

    # IMPLEMENT HERE
    text = prediction.strip()
    #check for GSM8K-style #### marker
    if "####" in text:
        text = text.split("####")[-1].strip()
    elif "###" in text:
        text = text.split("###")[-1].strip()

    #for my_prompt, look for "Final Answer: <number>"
    if prompt_name == "my_prompt":
        m = re.search(r"Final Answer:\s*(-?\d[\d,]*)", text, re.IGNORECASE)
        if m:
            return m.group(1).replace(",", "")

    #extract last number found in the text
    nums = re.findall(r"-?\d[\d,]*", text)
    if nums:
        return nums[-1].replace(",", "")
    return text.strip()

In [ ]:
# Do not change, method to calculate accuracy from predictions and ground truth labels

def evaluate_accuracy(predictions: List[str], ground_truths: List[str]) -> float:
    correct = 0
    total = len(predictions)

    for pred, true in zip(predictions, ground_truths):
        if pred == true:
            correct += 1

    accuracy = correct / total
    return accuracy * 100

### Next, write the wrapper function where you use all the building blocks constructed above to prompt the Gemini model (5 + 10 pts)


On how to prompt Gemini, refer to the [Gemini Text Generation Handbook](https://ai.google.dev/gemini-api/docs/text-generation?lang=python).

Hint: Reading this will help you figure out how to generate multiple candidates to implement Self-Consistency.

In [ ]:
def pipeline_generate(
    model_instance: Any,
    test_set: Dataset,
    prompt_function: Callable[..., str],
    process_answer_function: Callable[[str, Any], str],
    evaluation_function: Callable[[List[str], List[str]], float],
    self_consistency: int,
) -> float:
    """
    Args:
    model_instance (Any): The Google Gemini model instance.
    test_set (Dataset): The GSM8K test set to evaluate on.
    prompt_function (Callable): Function to generate prompts for the test set.
    process_answer_function (Callable): Function to process the model's generated answers.
    evaluation_function (Callable): Function to evaluate model's answers against the ground truth.
    self_consistency: Number of samples to run self-consistency approach on.
    If negative, 0 or 1, this implies regular prompting

    Returns:
    float: The accuracy of the model on the test set.
    """
    import time
    from collections import Counter
    from datasets import load_dataset

    # IMPLEMENT HERE
    train_raw = load_dataset("gsm8k", "main")["train"]
    train_proc = process_gsm8k_answers(train_raw)

    predictions: List[str] = []
    ground_truths: List[str] = []

    for i in range(len(test_set)):
        row = test_set[i]
        q = row["question"]
        true = str(row["processed_answer"]).strip()

        try:
            if self_consistency is not None and self_consistency > 1:
                votes: List[str] = []
                for _ in range(self_consistency):
                    pr = prompt_generation_zero_shot_cot(q)
                    resp = model_instance.generate_content(pr)
                    text = getattr(resp, "text", "") or ""
                    votes.append(process_answer_function(text, prompt_generation_zero_shot_cot))
                    time.sleep(2)
                pred = Counter(votes).most_common(1)[0][0]
            else:
                name = prompt_function.__name__
                if name in ("prompt_generation_5_shot", "prompt_generation_5_shot_cot"):
                    pr = prompt_function(q, train_proc)
                else:
                    pr = prompt_function(q)
                time.sleep(7)
                resp = model_instance.generate_content(pr)
                text = getattr(resp, "text", "") or ""
                pred = process_answer_function(text, prompt_function)

        except Exception as e:
            print(f"Error on sample {i}: {e}")
            pred = ""

        predictions.append(str(pred).strip())
        ground_truths.append(true)
        print(f"Sample {i+1}/{len(test_set)} — pred: {pred}, true: {true}")

    return evaluation_function(predictions, ground_truths)

In [ ]:
gsm8k_test_processed = process_gsm8k_answers(dataset['test'])

# The following line is just to test your systems, comment this line out to report results on the entire test set in 3.3
gsm8k_test_processed = Dataset.from_dict(gsm8k_test_processed[:5])

# Run model generation with zero-shot prompt generation
accuracy = pipeline_generate(
    model_instance=model,
    test_set=gsm8k_test_processed,
    prompt_function=prompt_generation_5_shot_cot, # Change this to test different prompt methods
    process_answer_function=answer_processing,
    evaluation_function=evaluate_accuracy,
    self_consistency=3,
)

print(f"Accuracy: {accuracy}%")

## 4.3 Complete this table based on your implementation in 3.2 and answer the following questions (5 + 5 pts)

### Round each value up to two decimal points (5 pts)

Fill after running `pipeline_generate` in the cell below for each `prompt_function` (and `self_consistency` for the last row). Use `gsm8k_test_processed` on the full test set when reporting final numbers.

Method|Accuracy
---|---|
0-shot|20%
0-shot CoT|80%
5-shot|33.33%
5-shot CoT|100%
My prompt|100%
0-shot CoT Self-Consistency|100%

### What was the intuition behind the prompt that you designed? (2 pts)

My prompt's approach was to be direct, strict, and very intentional with what is being asked. first I assign the model a role "You are an expert math tutor" which prepares the model to respond in a careful, step-by-step teaching style rather than just guessing an answer. Second, it provides explicit structured instructions: identify key numbers, write the main equation, compute step by step, and end with a clearly formatted answer which will guide the model into designing its own "plan" to answering the question to improve its ability to get it right. This combination resulted in 100% accuracy on our sample, matching 5-shot CoT performance.

### What are the merits and demerits of using advanced prompting approaches like Chain of Thought or Self-Consistency? (3 pts)

The merits of using advanced prompting approaches like Chain of Thought significantly improves accuracy on multi-step math. My results show a jump from 20% (0-shot) to 80% (0-shot CoT) just by asking the model to reason step by step as well as from 33.33$ (5-shot) to 100% (5-shot CoT), with no additional training required. 5-shot CoT and self-consistency both achieved 100%, showing that combining in-context examples with reasoning chains and majority voting produces the strongest results.

The demerits of this approach was that it was more expensive in terms of API calls and tokens. Self-consistency caused rate limit errors during this assignment. CoT also produces much longer outputs, increasing latency and cost. Also, on a small sample of 3-5 examples the accuracy numbers are noisy as a single correct or incorrect answer shifts accuracy by 33%, so these results should be interpreted cautiously.